# Gather checkpoints (CPU): one package of every finished run

Copies every finished run found in **Input** (best checkpoint without optimiser state, `config.yaml`,
`summary.json`) and every validation prediction file into `/kaggle/working/package`, with a manifest of sha256.
Test predictions are skipped. A run found twice must be identical, otherwise the notebook stops.
Used twice, as two separate notebooks:

1. `ai-pictionary-gather-pretrained`: Input = `ai-pictionary-step4a-output` pinned to the version that holds
   `runs/pretrained_v1` (the output of AI_pictionary version 11).
2. `ai-pictionary-checkpoints`: Input = the three collector notebooks (`ai-pictionary-collect-v1-leak`,
   `-collect-anytime-v1`, `-collect-v7-followup1`), `ai-pictionary-step4a-output` pinned to **version 4**,
   the output of `ai-pictionary-gather-pretrained`, and the output of `AI_pictionary` (latest: convergence_v1).
   Its output is the package for submission and the checkpoint input of final_v2.

**Accelerator: None** (CPU), **Internet on** (for the code). **Save Version → Save & Run All**.


In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Trung4n/AI-dictionary.git"
BRANCH = "pipeline-step1"
REPO = Path("/kaggle/working/repo")

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    print("No GITHUB_TOKEN secret: cloning without one (works for a public repository).")

url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
os.chdir("/kaggle/working")  # a re-run may still be inside the folder deleted next
if REPO.exists():
    shutil.rmtree(REPO)
done = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO)], capture_output=True, text=True)
if done.returncode != 0:
    message = done.stderr.replace(token, "***") if token else done.stderr
    raise RuntimeError(f"clone failed:\n{message}")
# Keep the token out of files that Kaggle may save with the notebook output.
subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
%cd /kaggle/working/repo
!pip install -q pytest
tests = subprocess.run("PYTHONPATH=src python -m pytest tests -q -k 'not processed_data'", shell=True, capture_output=True, text=True)
print(tests.stdout[-1500:])
if tests.returncode != 0:
    raise RuntimeError("tests failed: stopping")


In [ ]:
!PYTHONPATH=src python -m quickdraw_exp.gather --inputs /kaggle/input --out /kaggle/working/package --expect configs/final_v2.yaml
